In [ ]:
import os
import glob
import torch
from PIL import Image
from transformers import pipeline
from tqdm import tqdm
import numpy as np

def perform_depth_estimation(input_folder, output_folder, model_name='depth-anything/Depth-Anything-V2-Small-hf'):
    # Check for CUDA availability
    device = 0 if torch.cuda.is_available() else -1

    # Initialize the depth estimation pipeline
    depth_estimator = pipeline(task="depth-estimation", model=model_name, device=device)

    # Create the output directory if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Get list of .jpg images in the input folder
    image_paths = glob.glob(os.path.join(input_folder, '*.jpg'))

    # Process each image with a progress bar
    for img_path in tqdm(image_paths, desc="Processing images"):
        try:
            # Load the image
            image = Image.open(img_path)

            # Perform depth estimation
            result = depth_estimator(image)
            depth_map = result['depth']

            # Normalize depth map to 0-255 and convert to uint8
            depth_map = np.array(depth_map)
            norm_depth = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min())
            norm_depth = (norm_depth * 255).astype('uint8')

            # Convert to PIL Image
            depth_image = Image.fromarray(norm_depth)

            # Create new filename with 'depth_' prefix
            base_filename = os.path.basename(img_path)
            new_filename = f"depth_{base_filename}"
            output_path = os.path.join(output_folder, new_filename)
            
            # Save the depth map image
            depth_image.save(output_path)
        except Exception as e:
            print(f"Error processing {img_path}: {e}")

if __name__ == "__main__":
    input_folder = 'images_folder'
    output_folder = 'depth_predictions_DAV2'
    perform_depth_estimation(input_folder, output_folder)


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

c:\Users\sheri\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sheri\.cache\huggingface\hub\models--depth-anything--Depth-Anything-V2-Small-hf. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/99.2M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cpu
Processing images: 100%|██████████| 458/458 [01:42<00:00,  4.49it/s]
